# انحدار شجرة القرار (CART) — Google Colab

**الهدف:** التنبؤ بقيمة مستهدفة مستمرة باستخدام **انحدار شجرة القرار (CART)** — يلتقط **أنماطًا غير خطية** مع قواعد سهلة القراءة.

| مثال | المتغير(ات) (X) | الهدف (y) | مجموعة البيانات |
|---------|----------------|------------|---------|
| **المثال 1** | Level | Salary | `../Datasets/Position_Salaries.csv` |
| **المثال 2** | Area_sqft, Bedrooms, Age_years | Price | `../Datasets/house_price.csv` |

| المرحلة | الموضوع | الخلايا |
|-------|-------|-------|
| المرحلة 0 | الإعداد | التثبيت والاستيراد |
| — | دليل الخوارزمية | تعريفات CART، MSE، بنية الشجرة |
| المرحلة 1 | معالجة البيانات | تحميل ← تنظيف ← ترميز ← تقسيم |
| المرحلة 2 | الخوارزمية | تدريب ← تنبؤ ← تصور ← تقييم |

> **التشغيل:** بيئة التشغيل ← تشغيل الكل (أو Ctrl+F9)


---
# دليل الخوارزمية — انحدار شجرة القرار (CART)

## ما هي CART؟

**CART** = **Classification And Regression Trees** (أشجار التصنيف والانحدار).  
في الانحدار، تقسم الشجرة البيانات **لتقليل MSE** وتتنبأ **بالمتوسط** لـ y في كل ورقة.

## بنية الشجرة

| الجزء | الاسم | الدور |
|------|------|------|
| **الجذر** | عقدة الجذر | أول تقسيم — أعلى الشجرة |
| **العقدة الداخلية** | فرع | قرار: `feature ≤ threshold?` |
| **الورقة** | عقدة طرفية | التنبؤ النهائي = **mean(y)** للعينات في تلك الورقة |

## كيف يعمل التقسيم (معيار MSE)

في كل عقدة، تجرب CART كل متغير وكل عتبة للعثور على التقسيم الذي **يقلل MSE بأقصى قدر**:

**MSE (متوسط مربع الخطأ):**

`MSE = (1/n) · Σ(yᵢ − ȳ)²`

**درجة التقسيم:** اختر التقسيم ذا **أكبر تقليل في MSE** (خوارزمية جشعة).

## قاعدة التنبؤ

1. ابدأ من عقدة **الجذر**.
2. اتبع الفروع: `if X ≤ threshold` ← يسار، وإلا ← يمين.
3. عند الوصول إلى **ورقة**، تنبأ بـ `ŷ = mean(y)` لعينات التدريب في تلك الورقة.

## المعاملات الفائقة الرئيسية

| المعامل | الدور | التأثير |
|-----------|------|--------|
| `max_depth` | أقصى عمق للشجرة | ضحلة = أبسط، عميقة = فرط ملاءمة |
| `min_samples_split` | الحد الأدنى للعينات لتقسيم عقدة | أعلى = شجرة أبسط |
| `min_samples_leaf` | الحد الأدنى للعينات في ورقة | أعلى = تنبؤات أكثر سلاسة |
| `max_leaf_nodes` | الحد الأقصى لعدد الأوراق | يحدّ من تعقيد الشجرة |
| `random_state` | بذرة عشوائية | نتائج قابلة للتكرار |

## CART مقابل نماذج أخرى

| | الانحدار الخطي | شجرة القرار (CART) |
|---|-------------------|----------------------|
| الشكل | خط مستقيم / مستوى | **متدرج على شكل خطوات**، غير خطي |
| التحجيم | مطلوب أحيانًا | **غير مطلوب** |
| قابلية التفسير | المعاملات | **قواعد مرئية** (إذا كانت الشجرة صغيرة) |
| فرط الملاءمة | منخفض (نموذج بسيط) | **مرتفع** إذا كانت الشجرة عميقة جدًا |
| متعدد المتغيرات | نعم | نعم — تقسيم على أي متغير |

## ما يجب أن يتذكره الطالب

1. تستخدم CART **MSE** لاختيار التقسيمات في الانحدار.
2. تنبؤ الورقة = **متوسط** قيم y في تلك المنطقة.
3. **لا حاجة لتحجيم المتغيرات** في أشجار القرار.
4. تحكم في **`max_depth`** لتجنب فرط الملاءمة.
5. تنشئ الأشجار **دوال خطوات** — التنبؤات ثابتة داخل كل منطقة.


## المرحلة 0 — الخلية 0: تثبيت المكتبات

يتضمن Google Colab عادةً معظم المكتبات. تضمن هذه الخلية توفر الحزم المطلوبة.

**ما تفعله هذه الخلية:** تثبت scikit-learn و pandas و matplotlib و numpy و seaborn بصمت.


In [ ]:
# تثبيت المكتبات المطلوبة بصمت (-q يخفي المخرجات)
!pip install -q scikit-learn pandas matplotlib numpy seaborn


## المرحلة 0 — الخلية 1: استيراد المكتبات

استيراد المكتبات لمعالجة البيانات والتحضير والنمذجة والتقييم.

**ما تفعله هذه الخلية:** تحمّل numpy و pandas و matplotlib و sklearn DecisionTreeRegressor والمقاييس.


In [ ]:
# --- استيراد المكتبات ---
import numpy as np              # عمليات وأمصاف رقمية
import pandas as pd             # تحميل ومعالجة البيانات الجدولية
import matplotlib.pyplot as plt # إنشاء الرسوم البيانية
import seaborn as sns           # تصورات إحصائية (تنسيق اختياري)

from sklearn.model_selection import train_test_split       # تقسيم البيانات إلى تدريب/اختبار
from sklearn.impute import SimpleImputer                   # ملء القيم المفقودة
from sklearn.tree import DecisionTreeRegressor, plot_tree  # نموذج CART وتصور الشجرة
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score  # مقاييس التقييم

plt.rcParams['figure.figsize'] = (10, 6)  # حجم الرسم الافتراضي: عرض=10، ارتفاع=6 بوصة
plt.rcParams['font.family'] = ['Segoe UI', 'Tahoma', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid')            # خلفية بيضاء نظيفة مع خطوط شبكة
np.random.seed(42)                          # تثبيت البذرة العشوائية لتقسيمات قابلة للتكرار

print('المكتبات جاهزة')                      # تأكيد تحميل جميع الاستيرادات بنجاح


---
# المثال 1: رواتب المناصب — انحدار شجرة القرار

التنبؤ بـ **Salary** من **Level** الوظيفي. العلاقة **غير خطية** — تنشئ CART مناطق على شكل خطوات.

| العمود | الدور | الوصف |
|--------|------|-------------|
| `Level` | متغير (X) | مستوى الوظيفة (1–10) |
| `Salary` | الهدف (y) | الراتب السنوي بالدولار الأمريكي |

**الملف:** `../Datasets/Position_Salaries.csv`


---
# المرحلة 1: معالجة البيانات

تحضير البيانات قبل التدريب — نفس القالب يُعاد استخدامه لخوارزميات أخرى.


## المثال 1 — الخلية 1: تحميل واستكشاف البيانات

تحميل ملف CSV وإجراء استكشاف أولي (head، info، describe، shape).

**ما تفعله هذه الخلية:** تقرأ `../Datasets/Position_Salaries.csv` وتعرض إحصائيات أساسية.


In [ ]:
# الخطوة 1) تحميل مجموعة البيانات
dataset = pd.read_csv('../Datasets/Position_Salaries.csv')  # قراءة CSV إلى DataFrame

FEATURE_COL = 'Level'   # المتغير المستقل (X) — مستوى الوظيفة
TARGET_COL = 'Salary'   # المتغير التابع (y) — الراتب المراد التنبؤ به

print('أول 5 صفوف:')          # طباعة تسمية للجدول أدناه
display(dataset.head())         # عرض أول 5 صفوف لفحص البيانات

print('\nمعلومات مجموعة البيانات:')       # طباعة تسمية لأنواع الأعمدة وعدد القيم الفارغة
dataset.info()                  # عرض أسماء الأعمدة وأنواع البيانات وعدد القيم غير الفارغة

print('\nملخص إحصائي:')  # طباعة تسمية للإحصائيات الرقمية
display(dataset.describe())       # عرض العدد والمتوسط والانحراف المعياري والحد الأدنى والأقصى والربيعيات

print(f'\nالشكل: {dataset.shape[0]} صف × {dataset.shape[1]} عمود')  # إجمالي الصفوف والأعمدة


## المثال 1 — الخلية 2: تنظيف البيانات (معالجة القيم المفقودة)

التحقق من القيم المفقودة، إزالة التكرارات، وتطبيق الاستكمال إذا لزم الأمر.

**ما تفعله هذه الخلية:** تنظف مجموعة البيانات قبل النمذجة.


In [ ]:
# الخطوة 2) تنظيف البيانات

print('القيم المفقودة لكل عمود:')  # طباعة تسمية
print(dataset.isnull().sum())        # عد قيم NaN في كل عمود

rows_before = len(dataset)                              # تخزين عدد الصفوف قبل التنظيف
dataset = dataset.drop_duplicates().reset_index(drop=True)  # إزالة الصفوف المكررة
rows_after = len(dataset)                               # تخزين عدد الصفوف بعد إزالة التكرار
print(f'\nالتكرارات المحذوفة: {rows_before - rows_after}')  # عرض عدد التكرارات

num_cols = dataset.select_dtypes(include=[np.number]).columns.tolist()  # الأعمدة الرقمية فقط
imputer = SimpleImputer(missing_values=np.nan, strategy='mean')  # ملء NaN بمتوسط العمود
if dataset.isnull().sum().sum() > 0:               # إذا وُجدت قيم مفقودة
    dataset[num_cols] = imputer.fit_transform(dataset[num_cols])  # استكمال الأعمدة الرقمية
    print('تم استكمال القيم المفقودة بالمتوسط')      # تأكيد الاستكمال
else:
    print('لا توجد قيم مفقودة — لم يُطبَّق المُكمِّل')  # تخطي عند اكتمال البيانات

print(f'\nالصفوف بعد التنظيف: {rows_after}')    # العدد النهائي للصفوف


## المثال 1 — الخلية 3: ترميز البيانات الفئوية

نستخدم `Level` (رقمي). عمود `Position` نصي — يُتخطى لأن Level يُرمّز المرتبة بالفعل.

**ما تفعله هذه الخلية:** تتحقق من الأعمدة الفئوية وترمّزها إذا لزم الأمر.


In [ ]:
# الخطوة 3) الترميز الفئوي

cat_cols = dataset.select_dtypes(include=['object', 'category']).columns.tolist()  # إيجاد الأعمدة النصية
print(f'الأعمدة الفئوية (غير مستخدمة كـ X): {cat_cols}')  # أسماء المناصب — للمعلومات فقط
print(f'المتغير المستخدم في CART: {FEATURE_COL}')              # Level رقمي — لا حاجة للترميز
print('لا يلزم ترميز — X رقمي.')


## المثال 1 — الخلية 4: تقسيم البيانات

تعريف X (Level) و y (Salary)، ثم تقسيم 80/20.

**ما تفعله هذه الخلية:** تنشئ مصفوفات المتغيرات/الهدف وتطبّق train_test_split.


In [ ]:
# الخطوة 4) تقسيم التدريب-الاختبار

X = dataset[[FEATURE_COL]].values  # مصفوفة المتغيرات: Level (مصفوفة ثنائية الأبعاد لـ sklearn)
y = dataset[TARGET_COL].values     # متجه الهدف: قيم Salary

X_train, X_test, y_train, y_test = train_test_split(
    X, y,                  # البيانات المراد تقسيمها
    test_size=0.2,         # 20% اختبار، 80% تدريب
    random_state=42        # تقسيم قابل للتكرار
)

print(f'شكل X_train: {X_train.shape}')  # شكل متغيرات التدريب
print(f'شكل X_test:  {X_test.shape}')   # شكل متغيرات الاختبار
print(f'شكل y_train: {y_train.shape}')  # شكل أهداف التدريب
print(f'شكل y_test:  {y_test.shape}')   # شكل أهداف الاختبار


> **ملاحظة:** انحدار شجرة القرار **لا يتطلب** تحجيم المتغيرات. الأشجار تقسم وفق عتبات — المقياس لا يؤثر.


---
# المرحلة 2: انحدار شجرة القرار (CART)

تدريب نموذج CART للتنبؤ بـ Salary من Level.


## المثال 1 — الخلية 5: تدريب النموذج

تدريب `DecisionTreeRegressor` مع `max_depth=4` لتجنب فرط الملاءمة على بيانات صغيرة.

**ما تفعله هذه الخلية:** يُلائم نموذج CART ويطبع عمق الشجرة وعدد الأوراق.


In [ ]:
# الخطوة 5) تدريب مُرَجِّس شجرة القرار (CART)

regressor = DecisionTreeRegressor(
    max_depth=4,           # تحديد العمق لمنع فرط الملاءمة على 10 عينات
    min_samples_leaf=1,    # الحد الأدنى للعينات المطلوبة في عقدة ورقة
    random_state=42        # بنية شجرة قابلة للتكرار
)

regressor.fit(X_train, y_train)  # بناء الشجرة: إيجاد أفضل تقسيمات MSE على بيانات التدريب

print('تم تدريب شجرة القرار (CART) بنجاح.')  # تأكيد اكتمال التدريب
print(f'عمق الشجرة: {regressor.get_depth()}')       # العمق الفعلي للشجرة المبنية
print(f'عدد الأوراق: {regressor.get_n_leaves()}')  # إجمالي عقد الأوراق (مناطق التنبؤ)


## المثال 1 — الخلية 6: التنبؤ

التنبؤ بـ Salary على مجموعة الاختبار.

**ما تفعله هذه الخلية:** تولّد تنبؤات بتوجيه كل عينة إلى ورقة.


In [ ]:
# الخطوة 6) التنبؤ

y_pred_train = regressor.predict(X_train)  # التنبؤ بـ Salary لبيانات التدريب
y_pred_test = regressor.predict(X_test)    # التنبؤ بـ Salary لبيانات الاختبار

print('عينة من التنبؤات (مجموعة الاختبار):')  # طباعة تسمية
for i in range(len(y_test)):             # عرض جميع تنبؤات الاختبار
    print(f'  Level={X_test[i][0]:.0f} -> فعلي=${y_test[i]:,.0f}، متوقع=${y_pred_test[i]:,.0f}')


## المثال 1 — الخلية 7: التصور

رسم **منحنى التنبؤ على شكل خطوات** و**بنية الشجرة**.

**ما تفعله هذه الخلية:** يعرض تنبؤات CART المتدرجة ومخططًا مرئيًا للشجرة.


In [ ]:
# الخطوة 7) التصور — منحنى الخطوات + مخطط الشجرة

fig, axes = plt.subplots(1, 2, figsize=(16, 6))  # رسمتان فرعيتان: المنحنى والشجرة

# --- اليسار: منحنى دالة الخطوات ---
X_plot = np.linspace(X.min(), X.max(), 500).reshape(-1, 1)  # 500 نقطة لخطوات سلسة
y_plot = regressor.predict(X_plot)                            # تنبؤات CART (دالة خطوات)

axes[0].scatter(X_train, y_train, color='blue', label='تدريب', s=80, zorder=3)  # نقاط التدريب
axes[0].scatter(X_test, y_test, color='green', label='اختبار', s=80, zorder=3)       # نقاط الاختبار
axes[0].plot(X_plot, y_plot, color='red', linewidth=2, label='تنبؤ CART')    # منحنى على شكل خطوات
axes[0].set_xlabel('Level')             # تسمية المحور السيني
axes[0].set_ylabel('Salary (USD)')     # تسمية المحور الصادي
axes[0].set_title('دالة خطوات CART — رواتب المناصب')  # عنوان الرسم الفرعي
axes[0].legend()                        # عرض وسيلة الإيضاح

# --- اليمين: مخطط الشجرة ---
plot_tree(
    regressor,                          # نموذج CART المدرب
    feature_names=[FEATURE_COL],        # الاسم المعروض على عقد التقسيم
    filled=True,                        # تلوين العقد حسب القيمة
    rounded=True,                       # صناديق عقد مستديرة
    fontsize=9,                         # حجم الخط للقراءة
    ax=axes[1]                          # الرسم على الرسم الفرعي الثاني
)
axes[1].set_title('بنية شجرة CART')  # عنوان الرسم الفرعي

plt.tight_layout()  # ضبط المسافات
plt.show()          # عرض كلا الرسمين


## المثال 1 — الخلية 8: التقييم

تقييم CART بـ MAE و RMSE و R² على مجموعة الاختبار.

**ما تفعله هذه الخلية:** تحسب وتعرض مقاييس التقييم.


In [ ]:
# الخطوة 8) التقييم
mae = mean_absolute_error(y_test, y_pred_test)              # متوسط الخطأ المطلق بالدولار
rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))     # جذر متوسط مربع الخطأ
r2 = r2_score(y_test, y_pred_test)                          # التباين المفسَّر

results = pd.DataFrame({
    'Metric': ['MAE', 'RMSE', 'R²'],
    'Value': [mae, rmse, r2],
    'Description': [
        'متوسط الخطأ المطلق (USD)',
        'جذر متوسط مربع الخطأ (USD)',
        'معامل التحديد (1 = مثالي)'
    ]
})

display(results.round(4))  # عرض جدول المقاييس
print(f'\nR² اختبار المثال 1 = {r2:.4f}')  # طباعة ملخص R²


## لماذا تعمل CART لرواتب المناصب؟

| # | السبب | الشرح |
|---|--------|-------------|
| 1 | **قفزات راتب غير خطية** | يزداد الراتب على شكل خطوات في المستويات العليا — تقسيمات الشجرة تطابق ذلك |
| 2 | **لا حاجة للتحجيم** | الأشجار تقارن `Level ≤ threshold` — القيم الخام تعمل جيدًا |
| 3 | **مخرجات دالة خطوات** | كل ورقة تتنبأ براتب ثابت — يناسب القفزات المنفصلة |
| 4 | **قواعد مرئية** | `plot_tree` يعرض قواعد if/else دقيقة يمكن للطلاب قراءتها |
| 5 | **انتبه لفرط الملاءمة** | مع 10 صفوف فقط، استخدم `max_depth` للحد من حجم الشجرة |

> **الملخص:** تنشئ CART **مناطق** من Level مع تنبؤات راتب ثابتة — مثالية للأنماط على شكل خطوات.


---
# المثال 2: سعر المنزل — انحدار شجرة القرار

التنبؤ بـ **Price** من ثلاثة متغيرات للعقار باستخدام CART مع **متغيرات متعددة**.

| العمود | الدور | الوصف |
|--------|------|-------------|
| `Area_sqft` | متغير (X₁) | مساحة المعيشة بالقدم المربع |
| `Bedrooms` | متغير (X₂) | عدد غرف النوم |
| `Age_years` | متغير (X₃) | عمر المنزل بالسنوات |
| `Price` | الهدف (y) | سعر البيع بالدولار الأمريكي |

**الملف:** `../Datasets/house_price.csv`


## المثال 2 — الخلية 1: تحميل واستكشاف البيانات

تحميل CSV أسعار المنازل وفحص البيانات.

**ما تفعله هذه الخلية:** تقرأ `../Datasets/house_price.csv` وتعرض إحصائيات أساسية.


In [ ]:
# الخطوة 1) تحميل مجموعة البيانات
dataset = pd.read_csv('../Datasets/house_price.csv')  # قراءة CSV إلى DataFrame

FEATURE_COLS = ['Area_sqft', 'Bedrooms', 'Age_years']  # ثلاثة متغيرات مدخلة
TARGET_COL = 'Price'                                    # المتغير الهدف

print('أول 5 صفوف:')
display(dataset.head())

print('\nمعلومات مجموعة البيانات:')
dataset.info()

print('\nملخص إحصائي:')
display(dataset.describe())

print(f'\nالشكل: {dataset.shape[0]} صف × {dataset.shape[1]} عمود')


## المثال 2 — الخلية 2: تنظيف البيانات (معالجة القيم المفقودة)

تتضمن مجموعة البيانات هذه قيمًا مفقودة لعرض `SimpleImputer`.

**ما تفعله هذه الخلية:** تتحقق من القيم الفارغة، تزيل التكرارات، وتستكمل القيم المفقودة.


In [ ]:
# الخطوة 2) تنظيف البيانات

print('القيم المفقودة لكل عمود:')
print(dataset.isnull().sum())

rows_before = len(dataset)
dataset = dataset.drop_duplicates().reset_index(drop=True)
rows_after = len(dataset)
print(f'\nالتكرارات المحذوفة: {rows_before - rows_after}')

imputer = SimpleImputer(missing_values=np.nan, strategy='mean')
if dataset.isnull().sum().sum() > 0:
    dataset[FEATURE_COLS + [TARGET_COL]] = imputer.fit_transform(dataset[FEATURE_COLS + [TARGET_COL]])
    print('تم استكمال القيم المفقودة بالمتوسط')
else:
    print('لا توجد قيم مفقودة — لم يُطبَّق المُكمِّل')

print(f'\nالصفوف بعد التنظيف: {rows_after}')


## المثال 2 — الخلية 3: ترميز البيانات الفئوية

جميع الأعمدة رقمية — يُتخطى الترميز.

**ما تفعله هذه الخلية:** تؤكد أنه لا يلزم ترميز فئوي.


In [ ]:
# الخطوة 3) الترميز الفئوي

cat_cols = dataset.select_dtypes(include=['object', 'category']).columns.tolist()
if cat_cols:
    print(f'أعمدة فئوية موجودة: {cat_cols}')
else:
    print('لا توجد أعمدة فئوية — تم تخطي الترميز.')
    print(f'أعمدة المتغيرات: {FEATURE_COLS}')


## المثال 2 — الخلية 4: تقسيم البيانات

تعريف X (3 متغيرات) و y (Price)، ثم تقسيم 80/20.

**ما تفعله هذه الخلية:** تنشئ مصفوفات المتغيرات/الهدف وتطبّق train_test_split.


In [ ]:
# الخطوة 4) تقسيم التدريب-الاختبار

X = dataset[FEATURE_COLS].values  # مصفوفة المتغيرات: 3 أعمدة
y = dataset[TARGET_COL].values    # متجه الهدف: قيم Price

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'شكل X_train: {X_train.shape}')  # (n_train, 3)
print(f'شكل X_test:  {X_test.shape}')   # (n_test, 3)
print(f'شكل y_train: {y_train.shape}')
print(f'شكل y_test:  {y_test.shape}')


## المثال 2 — الخلية 5: تدريب النموذج

تدريب CART مع متغيرات متعددة — الشجرة تقسم على Area أو Bedrooms أو Age في كل عقدة.

**ما تفعله هذه الخلية:** يُلائم النموذج ويُبلغ عن بنية الشجرة.


In [ ]:
# الخطوة 5) تدريب مُرَجِّس شجرة القرار (CART)

regressor = DecisionTreeRegressor(
    max_depth=5,           # تحديد العمق للتعميم
    min_samples_split=4,   # يلزم 4 عينات على الأقل لتقسيم عقدة
    min_samples_leaf=2,    # كل ورقة يجب أن تحتوي على عينتين على الأقل
    random_state=42
)

regressor.fit(X_train, y_train)  # بناء الشجرة باستخدام معيار MSE على جميع المتغيرات الثلاثة

print('تم تدريب شجرة القرار (CART) بنجاح.')
print(f'عمق الشجرة: {regressor.get_depth()}')
print(f'عدد الأوراق: {regressor.get_n_leaves()}')

print('\nأهمية المتغيرات:')  # أي المتغيرات استُخدمت أكثر للتقسيم
for name, imp in zip(FEATURE_COLS, regressor.feature_importances_):
    print(f'  {name:15s} -> {imp:.4f}')  # أعلى = أكثر أهمية للتقسيمات


## المثال 2 — الخلية 6: التنبؤ

التنبؤ بأسعار المنازل على مجموعة الاختبار.

**ما تفعله هذه الخلية:** تولّد تنبؤات بتوجيه العينات عبر الشجرة.


In [ ]:
# الخطوة 6) التنبؤ

y_pred_train = regressor.predict(X_train)
y_pred_test = regressor.predict(X_test)

print('عينة من التنبؤات (مجموعة الاختبار):')
for i in range(min(5, len(y_test))):
    print(f'  فعلي=${y_test[i]:,.0f}، متوقع=${y_pred_test[i]:,.0f}')


## المثال 2 — الخلية 7: التصور

رسم **الفعلي مقابل المتوقع** و**أهمية المتغيرات**.

**ما تفعله هذه الخلية:** يُصوّر أداء النموذج وأي المتغيرات الأكثر أهمية.


In [ ]:
# الخطوة 7) التصور

fig, axes = plt.subplots(1, 2, figsize=(14, 5))  # رسمتان فرعيتان

axes[0].scatter(y_test, y_pred_test, color='green', alpha=0.7)  # فعلي مقابل متوقع
min_val = min(y_test.min(), y_pred_test.min())
max_val = max(y_test.max(), y_pred_test.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='تنبؤ مثالي')
axes[0].set_xlabel('السعر الفعلي (USD)')
axes[0].set_ylabel('السعر المتوقع (USD)')
axes[0].set_title('الفعلي مقابل المتوقع — سعر المنزل (CART)')
axes[0].legend()

axes[1].barh(FEATURE_COLS, regressor.feature_importances_, color='teal')  # مخطط أهمية المتغيرات
axes[1].set_xlabel('الأهمية')
axes[1].set_title('أهمية المتغيرات (CART)')

plt.tight_layout()
plt.show()


## المثال 2 — الخلية 8: التقييم

تقييم أداء CART بـ MAE و RMSE و R².

**ما تفعله هذه الخلية:** تحسب وتعرض مقاييس التقييم.


In [ ]:
# الخطوة 8) التقييم
mae = mean_absolute_error(y_test, y_pred_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
r2 = r2_score(y_test, y_pred_test)

results = pd.DataFrame({
    'Metric': ['MAE', 'RMSE', 'R²'],
    'Value': [mae, rmse, r2],
    'Description': [
        'متوسط الخطأ المطلق (USD)',
        'جذر متوسط مربع الخطأ (USD)',
        'معامل التحديد (1 = مثالي)'
    ]
})

display(results.round(4))
print(f'\nR² اختبار المثال 2 = {r2:.4f}')


## لماذا تعمل CART جيدًا لسعر المنزل؟

| # | السبب | الشرح |
|---|--------|-------------|
| 1 | **متغيرات متعددة** | CART تقسم تلقائيًا على Area و Bedrooms و Age |
| 2 | **تفاعلات غير خطية** | مثلًا مساحة كبيرة + غرف نوم كثيرة ← منطقة سعر أعلى |
| 3 | **أهمية المتغيرات** | تُظهر أي متغير يساهم أكثر في التقسيمات |
| 4 | **لا حاجة للتحجيم** | القدم المربع وعدد الغرف والعمر تعمل مباشرة |
| 5 | **R² مرتفع ممكن** | مع ضبط `max_depth`، CART تلائم أنماط سعر معقدة |

> **المقارنة:** CART مقابل الانحدار الخطي — الأشجار تلتقط **تأثيرات غير خطية** و**تفاعلية** دون هندسة متغيرات يدوية.
